In [ ]:
# Running this notebook refreshes data/unrate.csv from the live FRED API.
from pathlib import Path
import json
import os
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import urlopen

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv

current_path = Path.cwd().resolve()
project_root = next(
    (
        candidate
        for candidate in (current_path, *current_path.parents)
        if (candidate / "data").is_dir() and (candidate / "src").is_dir()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Run this notebook from within the project directory.")

notebooks_dir = project_root / "notebooks"
data_dir = project_root / "data"
env_path = project_root / ".env"
gitignore_path = project_root / ".gitignore"
previous_notebook_path = notebooks_dir / "05_api_json_fred_intro.ipynb"
output_csv_path = data_dir / "unrate.csv"

print("pandas_version:", pd.__version__)
print("project_root:", project_root)
print("notebooks_dir_exists:", notebooks_dir.exists())
print("data_dir_exists:", data_dir.exists())
print("previous_notebook_exists:", previous_notebook_path.exists())
print("env_file_exists:", env_path.exists())
print("gitignore_exists:", gitignore_path.exists())
print("output_csv_path:", output_csv_path)

In [ ]:
load_dotenv(dotenv_path=env_path, override=False)
fred_api_key = os.getenv("FRED_API_KEY")

invalid_key_values = {
    None,
    "",
    "replace_with_your_fred_api_key",
}

if fred_api_key in invalid_key_values:
    raise ValueError(
        "FRED_API_KEY is not configured. "
        "Set it in the local .env file before requesting data."
    )

print("fred_api_key_loaded:", True)

In [ ]:
fred_endpoint = "https://api.stlouisfed.org/fred/series/observations"
series_id = "UNRATE"
observation_start = "2019-01-01"
observation_end = "2024-12-31"
sort_order = "asc"

request_params = {
    "series_id": series_id,
    "api_key": fred_api_key,
    "file_type": "json",
    "observation_start": observation_start,
    "observation_end": observation_end,
    "sort_order": sort_order,
}

safe_request_params = {
    **request_params,
    "api_key": "***",
}

safe_request_text = json.dumps(safe_request_params, sort_keys=True)

print("fred_endpoint:", fred_endpoint)
print("safe_request_params:", safe_request_params)
print("safe_request_has_mask:", '"api_key": "***"' in safe_request_text)
print("actual_key_hidden_in_preview:", fred_api_key not in safe_request_text)

In [ ]:
request_query = urlencode(request_params)
request_url_with_key = f"{fred_endpoint}?{request_query}"

try:
    with urlopen(request_url_with_key, timeout=30) as response:
        response_status = response.status
        response_content_type = response.headers.get_content_type()
        response_text = response.read().decode("utf-8")
except HTTPError as error:
    raise RuntimeError(
        f"FRED returned HTTP {error.code}."
    ) from None
except URLError as error:
    raise RuntimeError(
        "Could not connect to FRED."
    ) from None

print("response_status:", response_status)
print("response_content_type:", response_content_type)
print("response_text_length:", len(response_text))

In [ ]:
fred_response_json = json.loads(response_text)
observations = fred_response_json.get("observations", [])

if not observations:
    raise ValueError(
        "FRED returned no observations. "
        "Check the series ID, date range, and API key."
    )

required_observation_keys = {"date", "value"}
all_observations_ready = all(
    required_observation_keys.issubset(observation)
    for observation in observations
)

print("response_python_type:", type(fred_response_json).__name__)
print("response_top_level_keys:", list(fred_response_json.keys()))
print("response_count:", fred_response_json.get("count"))
print("returned_observation_count:", len(observations))
print("all_observations_ready:", all_observations_ready)
print("first_observation:", observations[0])
print("last_observation:", observations[-1])

In [ ]:
raw_unrate_df = pd.DataFrame(observations)

print("raw_unrate_row_count:", len(raw_unrate_df))
print("raw_unrate_columns:", list(raw_unrate_df.columns))
print("raw_date_dtype:", raw_unrate_df["date"].dtype)
print("raw_value_dtype:", raw_unrate_df["value"].dtype)
raw_unrate_df.head()

In [ ]:
required_columns = {"date", "value"}
missing_required_columns = required_columns - set(raw_unrate_df.columns)

if missing_required_columns:
    raise KeyError(
        f"Missing required columns: {sorted(missing_required_columns)}"
    )

unrate_df = raw_unrate_df[["date", "value"]].rename(
    columns={"value": "unrate_pct"}
).copy()

unrate_df["date"] = pd.to_datetime(
    unrate_df["date"],
    errors="coerce",
)
unrate_df["unrate_pct"] = pd.to_numeric(
    unrate_df["unrate_pct"],
    errors="coerce",
)

unrate_df = unrate_df.sort_values("date").reset_index(drop=True)

print("date_dtype:", unrate_df["date"].dtype)
print("unrate_dtype:", unrate_df["unrate_pct"].dtype)
print("unrate_row_count:", len(unrate_df))
unrate_df.head()

In [ ]:
unit = "Percent"
frequency = "Monthly"
seasonal_adjustment = "Seasonally Adjusted"
source_name = "U.S. Bureau of Labor Statistics"
source_note = "FRED series UNRATE"

unrate_df["series_id"] = series_id
unrate_df["unit"] = unit
unrate_df["frequency"] = frequency
unrate_df["seasonal_adjustment"] = seasonal_adjustment
unrate_df["source_name"] = source_name
unrate_df["source_note"] = source_note

unrate_columns = [
    "date",
    "series_id",
    "unrate_pct",
    "unit",
    "frequency",
    "seasonal_adjustment",
    "source_name",
    "source_note",
]

unrate_df = unrate_df[unrate_columns]

print("unrate_column_count:", len(unrate_df.columns))
print("unrate_columns:", list(unrate_df.columns))
unrate_df.head()

In [ ]:
expected_monthly_dates = pd.date_range(
    start=observation_start,
    end=observation_end,
    freq="MS",
)
expected_observation_count = len(expected_monthly_dates)

actual_dates = set(unrate_df["date"].dropna())
expected_dates = set(expected_monthly_dates)
missing_expected_dates = sorted(expected_dates - actual_dates)

print("expected_observation_count:", expected_observation_count)
print("actual_observation_count:", len(unrate_df))
print("row_count_match:", len(unrate_df) == expected_observation_count)
print("date_missing_count:", unrate_df["date"].isna().sum())
print("value_missing_count:", unrate_df["unrate_pct"].isna().sum())
print("duplicate_date_count:", unrate_df["date"].duplicated().sum())
print("missing_expected_date_count:", len(missing_expected_dates))
print("date_is_monotonic:", unrate_df["date"].is_monotonic_increasing)

In [ ]:
if not data_dir.exists():
    raise FileNotFoundError(
        "The project data directory is missing. Create it before running this cell."
    )

unrate_df.to_csv(
    output_csv_path,
    index=False,
    date_format="%Y-%m-%d",
)

print("saved_csv_path:", output_csv_path)
print("saved_csv_exists:", output_csv_path.exists())
print("saved_csv_name:", output_csv_path.name)
print("saved_csv_parent:", output_csv_path.parent.name)

In [ ]:
saved_unrate_df = pd.read_csv(
    output_csv_path,
    parse_dates=["date"],
)

print("saved_row_count:", len(saved_unrate_df))
print("saved_column_count:", len(saved_unrate_df.columns))
print("saved_columns:", list(saved_unrate_df.columns))
print("saved_first_date:", saved_unrate_df["date"].min().date())
print("saved_last_date:", saved_unrate_df["date"].max().date())
print("saved_value_missing_count:", saved_unrate_df["unrate_pct"].isna().sum())
saved_unrate_df.head()

In [ ]:
unrate_plot_df = saved_unrate_df.dropna(
    subset=["date", "unrate_pct"]
).set_index("date")

ax = unrate_plot_df["unrate_pct"].plot(
    kind="line",
    figsize=(10, 5),
)
ax.set_title("U.S. Unemployment Rate (UNRATE)")
ax.set_xlabel("date")
ax.set_ylabel("unrate_pct")

plt.tight_layout()
plt.show()

In [ ]:
analysis_df = saved_unrate_df.dropna(
    subset=["date", "unrate_pct"]
).sort_values("date").copy()

if analysis_df.empty:
    raise ValueError("No valid UNRATE observations are available.")

start_row = analysis_df.iloc[0]
end_row = analysis_df.iloc[-1]
peak_row = analysis_df.loc[analysis_df["unrate_pct"].idxmax()]

start_date = start_row["date"]
end_date = end_row["date"]
peak_date = peak_row["date"]
start_value = float(start_row["unrate_pct"])
end_value = float(end_row["unrate_pct"])
peak_value = float(peak_row["unrate_pct"])
change_value = end_value - start_value

if change_value > 0:
    direction_text = f"increased by {change_value:.1f} percentage points"
elif change_value < 0:
    direction_text = f"decreased by {abs(change_value):.1f} percentage points"
else:
    direction_text = "was unchanged"

interpretation_sentence = (
    f"From {start_date:%Y-%m} to {end_date:%Y-%m}, UNRATE "
    f"{direction_text}, from {start_value:.1f}% to {end_value:.1f}%. "
    f"The highest value in this period was {peak_value:.1f}% in {peak_date:%Y-%m}. "
    "These are monthly, seasonally adjusted unemployment rates from the "
    "U.S. Bureau of Labor Statistics, retrieved through FRED. "
    "The chart describes the change without establishing its cause."
)

print(interpretation_sentence)
